# 第2回 分解と予測評価

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：見えていない値を当てる方法は、どう比べればよいか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "fable", "feasts", "fda"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(tsibbledata)
library(feasts)
library(fable)
library(ggtime)   # 分解と予測の図（autoplot）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

### A-1. 値を隠して、埋める

測るはずだった値が抜けていることを、欠測（missing value）といいます。欠測を埋めることを、補間（interpolation）といいます。

埋め方の良し悪しを比べるために、欠測のない系列から値をわざと隠し、埋めた値を隠した値と比べます。

人工の系列を2本作ります。長さは364日（52週）です。

- `smooth`：なめらかな傾向 ＋ ノイズ
- `seasonal`：`smooth`に、曜日ごとの決まった上下（季節成分）を足したもの

In [ ]:
set.seed(1)
n <- 364
day <- 1:n
trend  <- 100 + 8 * sin(2 * pi * day / 364) + 0.02 * day    # なめらかな傾向
season <- rep(c(5, 6, 6, 5, 4, -12, -14), length.out = n)   # 7日でくり返す上下（合計は0）
noise  <- rnorm(n, sd = 2)                                  # ノイズ
sim <- tsibble(day = day, smooth = trend + noise, seasonal = trend + season + noise,
               index = day)
ggplot(sim, aes(day, smooth)) +
  geom_line() +
  labs(x = "Day", y = "Value (smooth series)")

埋める方法を3つ用意します。どれも、欠測を`NA`で表した系列を受け取り、埋めた系列を返します。

- 線形補間（linear interpolation）：欠測の両側の値を、直線で結ぶ。
- スプライン補間（spline interpolation）：すべての観測点を通る、なめらかな曲線（3次スプライン）で結ぶ。
- 関数データ解析（functional data analysis, FDA）による補間：系列を「なめらかな関数 ＋ ノイズ」と考える。基底関数（basis function）の重みつきの和でなめらかな関数を推定し、その関数の値で埋める。曲線は観測点を通らなくてよい。

誤差は、二乗平均平方根誤差（root mean squared error, RMSE）で測ります。

$$
\mathrm{RMSE} = \sqrt{\frac{1}{m} \sum_{t \in \text{隠した日}} \left( \hat{y}_t - y_t \right)^2}
$$

In [ ]:
# 線形補間
fill_linear <- function(y) {
  t <- seq_along(y); ok <- !is.na(y)
  approx(t[ok], y[ok], xout = t, rule = 2)$y
}
# スプライン補間（観測点をすべて通る3次スプライン）
fill_spline <- function(y) {
  t <- seq_along(y); ok <- !is.na(y)
  spline(t[ok], y[ok], xout = t, method = "natural")$y
}
# 誤差（RMSE）
rmse <- function(estimate, truth) sqrt(mean((estimate - truth)^2))

In [ ]:
# 関数データ解析：基底関数（Bスプライン、1週間に約1個）と粗さの罰則で、なめらかな関数を推定する
fda_fit <- function(y, lambda) {
  t <- seq_along(y); ok <- !is.na(y)
  basis <- fda::create.bspline.basis(range(t), nbasis = round(length(y) / 7) + 2)
  fda::smooth.basis(t[ok], y[ok], fda::fdPar(basis, 2, lambda))
}
# 推定した関数の、すべての日での値
fda_curve <- function(y, lambda) {
  as.numeric(fda::eval.fd(seq_along(y), fda_fit(y, lambda)$fd))
}
# 関数データ解析による補間（欠測の日だけを、推定した関数の値で埋める）
fill_fda <- function(y, lambda) {
  ifelse(is.na(y), fda_curve(y, lambda), y)
}

`smooth`の150日目から14日間を隠します。

In [ ]:
gap <- 150:163                          # 隠す日
y_true <- sim$smooth
y_obs  <- replace(y_true, gap, NA)      # 隠した系列（欠測はNA）
sum(is.na(y_obs))

関数データ解析では、曲線のなめらかさを決める数$\lambda$（罰則の強さ）を選ぶ必要があります。
$\lambda$が小さいと曲線は観測点を追いかけ、大きいと直線に近づきます。

ここでは、一般化交差検証（generalized cross-validation, GCV）の値が最も小さくなる$\lambda$を選びます。
GCVは「観測点を1つ隠し、残りから当てる」ことをすべての点でくり返したときの誤差を、近似する量です。

In [ ]:
# GCVが最小になるλを選ぶ
choose_lambda <- function(y, log10_lambda = seq(0, 8, 0.5)) {
  gcv <- sapply(log10_lambda, function(l) sum(fda_fit(y, 10^l)$gcv))
  list(lambda = 10^log10_lambda[which.min(gcv)],
       table = data.frame(log10_lambda = log10_lambda, gcv = gcv))
}

In [ ]:
sel <- choose_lambda(y_obs)
lambda <- sel$lambda
log10(lambda)
ggplot(sel$table, aes(log10_lambda, gcv)) +
  geom_line() + geom_point() +
  labs(x = "log10(lambda)", y = "GCV")

図を描く関数を用意します。隠した期間のまわりだけを描き、隠した値を黒い点で示します。

In [ ]:
# 埋めた値を、方法ごとの縦長の表にする（図のため）
long_fills <- function(fills, gap) {
  do.call(rbind, lapply(names(fills), function(m)
    data.frame(day = gap, method = m, value = fills[[m]][gap])))
}
# 隠した期間のまわりを描く
plot_fills <- function(y_true, fills, gap, margin = 21) {
  near <- max(1, min(gap) - margin):min(length(y_true), max(gap) + margin)
  ggplot() +
    geom_line(data = data.frame(day = near, value = replace(y_true, gap, NA)[near]),
              aes(day, value), colour = "grey50") +
    geom_point(data = data.frame(day = gap, value = y_true[gap]), aes(day, value)) +
    geom_line(data = long_fills(fills, gap), aes(day, value, colour = method), linewidth = 0.8) +
    geom_point(data = long_fills(fills, gap), aes(day, value, colour = method), size = 1) +
    labs(x = "Day", y = "Value", colour = "Method")
}

3つの方法で埋めて、隠した値と比べます。

In [ ]:
fills <- list(linear = fill_linear(y_obs),
              spline = fill_spline(y_obs),
              fda    = fill_fda(y_obs, lambda))
plot_fills(y_true, fills, gap)
sapply(fills, function(f) rmse(f[gap], y_true[gap]))

- スプライン補間は、大きく外れます。観測点をすべて通るので、ノイズまで追いかけ、その勢いのまま欠測の中でふくらみます。
- 線形補間と関数データ解析は、どちらも隠した値の真ん中あたりを通ります。この1か所では、誤差はほぼ同じです。

1か所だけでは、たまたまかもしれません。隠す場所と長さを変えて、くり返します。
長さは1日から21日まで、場所は16か所です。

In [ ]:
# 隠す場所と長さを変えて、方法ごとに誤差の平均を出す（methods は、埋める関数に名前を付けたリスト）
gap_study <- function(y, methods, lens = c(1, 2, 3, 5, 7, 10, 14, 21),
                      starts = seq(20, 320, by = 20)) {
  do.call(rbind, lapply(lens, function(len) {
    err <- sapply(starts, function(s) {
      g <- s:(s + len - 1)                   # 隠す日
      y_obs <- replace(y, g, NA)
      sapply(methods, function(f) rmse(f(y_obs)[g], y[g]))
    })
    data.frame(len = len, method = rownames(err), rmse = rowMeans(err))
  }))
}
# 結果の図
plot_study <- function(res) {
  ggplot(res, aes(len, rmse, colour = method)) +
    geom_line() + geom_point() +
    labs(x = "Gap length (days)", y = "RMSE (mean over 16 positions)", colour = "Method")
}

In [ ]:
lambda <- choose_lambda(sim$smooth)$lambda
methods <- list(linear = fill_linear,
                spline = fill_spline,
                fda    = function(y) fill_fda(y, lambda))
res_smooth <- gap_study(sim$smooth, methods)
plot_study(res_smooth)
round(xtabs(rmse ~ len + method, res_smooth), 2)

- 関数データ解析の誤差が、どの長さでも最も小さくなります。まわりの多くの点からなめらかな関数を推定するので、ノイズがならされるからです。
- 線形補間は、両端の2点だけで決まります。両端のノイズを、そのまま引きずります。
- スプライン補間は、隠す日数が増えるほど誤差が大きくなります。

### A-2. 季節のある系列では、3つとも外れる

同じことを、曜日ごとの上下がある`seasonal`で行います。隠す日は同じです。

In [ ]:
ggplot(sim, aes(day, seasonal)) +
  geom_line() +
  labs(x = "Day", y = "Value (seasonal series)")

In [ ]:
y_true <- sim$seasonal
y_obs  <- replace(y_true, gap, NA)
lambda <- choose_lambda(y_obs)$lambda
fills <- list(linear = fill_linear(y_obs),
              spline = fill_spline(y_obs),
              fda    = fill_fda(y_obs, lambda))
plot_fills(y_true, fills, gap)
sapply(fills, function(f) rmse(f[gap], y_true[gap]))

3つとも、曜日ごとの上下を再現できません。
どの方法も「欠測の中では、値はなめらかに変わる」と考えているからです。季節のある系列では、この考えが成り立ちません。

### A-3. 系列を、傾向・季節・残りに分ける

系列を3つの成分の和に分けることを、分解（decomposition）といいます。

$$
y_t = T_t + S_t + R_t
$$

$T_t$は傾向（trend）、$S_t$は季節成分（seasonal component）、$R_t$は残り（remainder）です。

STL分解（Seasonal and Trend decomposition using Loess）で、`seasonal`を分けます。季節の周期は7日です。

In [ ]:
dcmp <- sim |>
  model(STL(seasonal ~ season(period = 7, window = "periodic"))) |>
  components()
autoplot(dcmp) +
  labs(x = "Day")

この系列は自分で作ったので、本当の成分が分かっています。推定した成分と比べます。

In [ ]:
# 季節成分：最初の7日について、推定した値と本当の値
data.frame(day = 1:7, estimated = round(dcmp$season_7[1:7], 2), true = season[1:7])
# 傾向と残り：本当の値との差（RMSE）と、残りの標準偏差（本当は2）
c(trend_rmse = rmse(dcmp$trend, trend), remainder_sd = sd(dcmp$remainder))

季節成分と傾向を、ほぼ取り出せています。

### A-4. 季節成分を考慮した補間

分解を使うと、季節のある系列も埋められます。

1. 欠測を、いったん線形補間で埋める（STL分解は欠測があると計算できないため）
2. STL分解で、季節成分を取り出す
3. もとの系列から季節成分を引く（季節調整済みの系列。欠測は欠測のまま）
4. 季節調整済みの系列を、線形補間で埋める
5. 季節成分を足して戻す

In [ ]:
# 季節成分を考慮した補間（period は季節の周期）
fill_seasonal <- function(y, period = 7) {
  y_first <- fill_linear(y)                                            # 1
  s <- stl(ts(y_first, frequency = period), s.window = "periodic")     # 2（A-3と同じSTL分解）
  s <- as.numeric(s$time.series[, "seasonal"])
  adjusted <- y - s                                                    # 3
  fill_linear(adjusted) + s                                            # 4, 5
}

In [ ]:
fills$seasonal <- fill_seasonal(y_obs)
plot_fills(y_true, fills, gap)
sapply(fills, function(f) rmse(f[gap], y_true[gap]))

季節成分を考慮した補間だけが、曜日ごとの上下を再現します。

1か所だけでは、たまたまかもしれません。A-1と同じように、隠す場所と長さを変えてくり返します。

In [ ]:
lambda <- choose_lambda(sim$seasonal)$lambda
methods <- list(linear   = fill_linear,
                spline   = fill_spline,
                fda      = function(y) fill_fda(y, lambda),
                seasonal = fill_seasonal)
res_seasonal <- gap_study(sim$seasonal, methods)
plot_study(res_seasonal)
round(xtabs(rmse ~ len + method, res_seasonal), 2)

- 季節成分を考慮した補間は、隠す日数が増えても誤差がほとんど増えません。ノイズの標準偏差（2）と同じくらいです。
- スプライン補間は、隠す日数が増えるほど誤差が大きくなります。
- 関数データ解析は、季節成分をノイズとみなしてならすので、1日だけの欠測でも誤差が大きくなります。

### A-5. 予測は、系列の端にある欠測である

これから先の値は、まだ見えていません。予測（forecast）は、系列の右端の外にある欠測を埋めることにあたります。
補間と違い、欠測の右側に値がありません。

予測の良し悪しも、値を隠して比べます。系列の最後の28日を隠し、それより前だけを使って予測します。

- 訓練データ（training data）：予測を作るのに使う部分（1〜336日目）
- テストデータ（test data）：隠しておき、予測と比べる部分（337〜364日目）

In [ ]:
h <- 28   # 隠す日数
train <- sim |> filter(day <= n - h)
test  <- sim |> filter(day >  n - h)
c(train = nrow(train), test = nrow(test))

比べる基準として、単純な予測を3つ使います。これらをベンチマーク（benchmark）と呼びます。

- 平均法（mean method）：訓練データの平均を、先の値とする。
- ナイーブ法（naive method）：最後の値を、先の値とする。
- 季節ナイーブ法（seasonal naive method）：1周期前（7日前）の同じ曜日の値を、先の値とする。

In [ ]:
fit <- train |>
  model(mean   = MEAN(seasonal),
        naive  = NAIVE(seasonal),
        snaive = SNAIVE(seasonal ~ lag(7)))
fc <- fit |> forecast(h = h)
autoplot(fc, sim |> filter(day > n - 3 * h), level = NULL) +
  labs(x = "Day", y = "Value (seasonal series)", colour = "Method")

誤差は、RMSEと平均絶対誤差（mean absolute error, MAE）で測ります。

$$
\mathrm{MAE} = \frac{1}{h} \sum_{t \in \text{テスト}} \left| \hat{y}_t - y_t \right|
$$

In [ ]:
accuracy(fc, sim) |> select(.model, RMSE, MAE)

季節ナイーブ法の誤差が最も小さくなります。この系列は曜日ごとの上下が大きいので、それを再現できる方法が勝ちます。

### A-6. 隠し方は、時間の順を守る

機械学習では、データをランダムに訓練とテストに分けることがよくあります。時系列で同じことをすると、何が起きるかを確かめます。

長さ364の系列から28個を隠し、「直前の見えている値」で当てます。隠し方は2通りです。

- `random`：ランダムに28個を隠す。
- `time`：最後の28個を隠す。

これを、ホワイトノイズ（iid）とランダムウォーク（iidでない）で、500回ずつくり返します。

In [ ]:
set.seed(1)
one <- function(make) {
  x <- make(364)
  hide <- sort(sample(2:364, 28))                                   # ランダムに隠す
  prev <- sapply(hide, function(i) max(setdiff(1:(i - 1), hide)))   # 直前の見えている日
  c(random = rmse(x[prev], x[hide]),
    time   = rmse(rep(x[336], 28), x[337:364]))                     # 最後の28個を隠す
}
res_split <- rbind(
  white_noise = rowMeans(replicate(500, one(rnorm))),
  random_walk = rowMeans(replicate(500, one(function(n) cumsum(rnorm(n)))))
)
round(res_split, 2)

- ホワイトノイズ：2つの隠し方で、誤差はほぼ同じです。iidなら、どこを隠しても同じ問題になります。
- ランダムウォーク：ランダムに隠すと、誤差がずっと小さく出ます。隠した日のすぐ隣が見えているので、予測ではなく補間の問題になっているからです。

予測を評価するときは、時間の順を守って隠します。

1回の分け方だけでは、たまたまその期間で当たっただけかもしれません。
訓練データを少しずつ伸ばしながら、同じ評価をくり返します。これを時系列交差検証（time series cross-validation）といいます。

In [ ]:
cv <- sim |>
  filter(day <= n - h) |>
  stretch_tsibble(.init = n - 6 * h, .step = h)   # 訓練データを28日ずつ伸ばす
cv |> as_tibble() |> group_by(.id) |> summarise(train_to = max(day))
cv |>
  model(mean   = MEAN(seasonal),
        naive  = NAIVE(seasonal),
        snaive = SNAIVE(seasonal ~ lag(7))) |>
  forecast(h = h) |>
  accuracy(sim) |>
  select(.model, RMSE, MAE)

6回の評価をまとめても、季節ナイーブ法が最も良い、という順位は変わりません。

## B. 実データを解析する

`tsibbledata`パッケージの`vic_elec`を使います。オーストラリアのビクトリア州の電力需要です。

B節は、A節で作った関数（`fill_linear`など）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
series    <- "Demand"   # 系列："Demand"（1日の電力需要）か "Temperature"（1日の最高気温）
year_from <- 2014       # 始まりの年：2012〜2014
year_to   <- 2014       # 終わりの年：year_from 以上、2014 以下
gap_from  <- 150        # 隠し始める日（期間の最初の日を1とする通し番号）：30〜300
gap_len   <- 14         # 隠す日数：1〜28
h         <- 28         # 予測する日数：7, 14, 28 のどれか

### B-1. データを確かめる

In [ ]:
glimpse(vic_elec)
interval(vic_elec)
range(vic_elec$Date)
c(rows = nrow(vic_elec), missing = sum(is.na(vic_elec$Demand)), gaps = has_gaps(vic_elec)$.gaps)

- 1行は30分です。`Demand`が電力需要（MWh）、`Temperature`がメルボルンの気温（℃）です。
- 期間は2012年1月1日から2014年12月31日、52608行です。
- 欠測（`NA`）も、行の抜けもありません。

### B-2. 前処理

30分ごとの値を、1日ごとにまとめます。電力需要は1日の合計（GWh）、気温は1日の最高値にします。
そのあと、期間を絞り、設定で選んだ系列を`y`という名前にします。

In [ ]:
daily_all <- vic_elec |>
  index_by(Date) |>
  summarise(Demand = sum(Demand) / 1000,          # 1日の電力需要（GWh）
            Temperature = max(Temperature),       # 1日の最高気温（℃）
            Holiday = any(Holiday))               # 祝日かどうか
interval(daily_all)
daily <- daily_all |>
  filter(format(Date, "%Y") >= year_from, format(Date, "%Y") <= year_to) |>
  mutate(day = row_number(), y = .data[[series]])
y_lab <- if (series == "Demand") "Daily demand (GWh)" else "Daily max temperature (deg C)"
n <- nrow(daily)
n

In [ ]:
ggplot(daily, aes(Date, y)) +
  geom_line() +
  labs(x = "Date", y = y_lab)

### B-3. 分解

In [ ]:
dcmp <- daily |>
  model(STL(y ~ season(period = 7, window = "periodic"))) |>
  components()
autoplot(dcmp) +
  labs(x = "Date")

人工データでは、残りはノイズでした。実データの残りを、第1回のACFで調べます。

In [ ]:
dcmp |>
  ACF(remainder, lag_max = 28) |>
  autoplot() +
  labs(x = "Lag h (day)", y = "Sample ACF of remainder")

破線の外に出るラグがあります。傾向と季節を取り除いても、残りはホワイトノイズになっていません。

### B-4. 値を隠して、埋める

このデータに欠測はないので、わざと行を抜きます。

In [ ]:
gap <- gap_from:(gap_from + gap_len - 1)       # 隠す日
obs <- daily |> filter(!(day %in% gap))        # 行ごと抜く
has_gaps(obs)

行が抜けていると、`has_gaps()`が`TRUE`になります。値が`NA`になっているわけではないので、`is.na()`では見つかりません。
`fill_gaps()`で、抜けた日の行を作り、値を`NA`にします。

In [ ]:
obs <- obs |> fill_gaps()
c(rows = nrow(obs), missing = sum(is.na(obs$y)))

4つの方法で埋めます。

In [ ]:
y_true <- daily$y
y_obs  <- obs$y
lambda <- choose_lambda(y_obs)$lambda
fills <- list(linear   = fill_linear(y_obs),
              spline   = fill_spline(y_obs),
              fda      = fill_fda(y_obs, lambda),
              seasonal = fill_seasonal(y_obs))
plot_fills(y_true, fills, gap) +
  labs(x = "Day", y = y_lab)
gap_rmse <- sapply(fills, function(f) rmse(f[gap], y_true[gap]))
round(gap_rmse, 2)

### B-5. 予測を比べる

最後の`h`日を隠し、3つのベンチマークで予測します。

In [ ]:
train <- daily |> filter(day <= n - h)
fit <- train |>
  model(mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = SNAIVE(y ~ lag(7)))
fc <- fit |> forecast(h = h)
autoplot(fc, daily |> filter(day > n - 3 * h), level = NULL) +
  labs(x = "Date", y = y_lab, colour = "Method")

In [ ]:
acc <- accuracy(fc, daily) |> select(.model, RMSE, MAE)
acc

### B-6. 時系列交差検証

訓練データを`h`日ずつ伸ばしながら、同じ評価を6回くり返します。

In [ ]:
acc_cv <- daily |>
  filter(day <= n - h) |>
  stretch_tsibble(.init = n - 6 * h, .step = h) |>
  model(mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = SNAIVE(y ~ lag(7))) |>
  forecast(h = h) |>
  accuracy(daily) |>
  select(.model, RMSE, MAE)
acc_cv

読み方：

- B-4：季節成分を考慮した補間が最も良いとは限りません。隠した期間に祝日や猛暑の日が入ると、「いつもの曜日の上下」から外れるからです。
- B-5：最後の`h`日は年末です。祝日が続くので、季節ナイーブ法が外れることがあります。
- B-6：6回の評価をまとめると、1回だけの評価（B-5）と順位が変わることがあります。1回の結果だけで、方法の優劣を決めないでください。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `series` | `"Demand"` `"Temperature"` |
| `year_from` | 2012〜2014 |
| `year_to` | `year_from`以上、2014以下 |
| `gap_from` | 30〜300 |
| `gap_len` | 1〜28 |
| `h` | 7、14、28 |

読むときの観点：

1. 隠す日数（`gap_len`）を1、3、14、28と変えると、4つの補間の順位はどこで入れ替わるか。
2. 隠す場所（`gap_from`）を変えると、季節成分を考慮した補間の誤差はどう変わるか。誤差が大きい場所には、何があるか。
3. 系列を`"Temperature"`にすると、季節成分を考慮した補間と季節ナイーブ法は、まだ有利か。分解の図の季節成分の大きさと合わせて読む。
4. 予測する日数（`h`）を変えると、B-5とB-6でベンチマークの順位は変わるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（Demand、2014年、150日目から14日間、28日先）の結果と比べてください。